# Architecture Decisions

*Notebook 31*

Every pattern in this course adds capability — and complexity. This notebook gives you a decision framework for choosing the right level of complexity for each problem.
<br>
<br>

**Topics:**
- Model selection: mini vs reasoning vs parallel specialists

- Single agent vs multi-agent

- Tool-heavy vs instruction-heavy agents

- When memory and sessions add value vs add noise

- When MCP is worth the setup

- Avoiding unnecessary complexity

- Choosing the right pattern — summary table

- Course wrap-up and suggested next projects

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(dotenv_path=Path("..") / ".env")

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"


print("✅ Ready!")

---

## 🎯 The Problem

The hardest part of building agent systems is knowing when to stop adding patterns.

Every section here answers one question: does this problem actually need this level of complexity?

---

## 🧠 Part 1: Model Selection

The default should always be `MODEL` (`gpt-5-mini`).

It's fast, cheap, and handles the vast majority of agent tasks well.

Upgrade only when you have evidence the simpler model isn't working.

**Use `MODEL` for:**

- Tool selection and execution — following clear instructions to call the right tool

- Structured output generation — filling in a Pydantic model from provided context

- Single-turn tasks with well-defined inputs and outputs

- Most customer-facing agents where latency matters

**Use `REASONING_MODEL` for:**

- Orchestration — deciding which agents to call and in what order

- Tasks requiring multi-step planning before execution

- Judgment calls with significant ambiguity or tradeoffs

- Capstone-style tasks where quality matters more than speed

**Use parallel specialists for:**

- Independent subtasks that can run concurrently (Parallel Execution pattern)

- When one model making all decisions creates a bottleneck

- Research pipelines where multiple angles improve final quality

If one subtask needs the output of another, it's not a good parallel-specialist candidate — run those sequentially instead.

Model upgrades are the complexity decision people reach for too early.

Always run your Lesson 09 test set on both models and compare quality, latency, and cost before committing.

In [ ]:
# Model comparison pattern — run before upgrading
# ------------------------------------------------
MODEL_EVAL_TEMPLATE = '''
from agents import Agent, Runner
from config import MODEL, REASONING_MODEL

# Your golden test set from Lesson 09
test_cases = [
    {"input": "...", "expected": "..."},
    # add more cases
]

async def eval_model(model: str) -> dict:
    """Run test set against one model, return quality and latency."""
    import time
    results = []

    agent = Agent(name="TestAgent", instructions="...", model=model)

    for case in test_cases:
        start = time.time()

        result = await Runner.run(agent, input=case["input"])

        elapsed = time.time() - start
        results.append({
            "output": result.final_output,
            "latency": elapsed,
            # score with judge agent (Lesson 09 pattern)
        })

    return results


# Compare before committing to REASONING_MODEL
# mini_results = await eval_model(MODEL)
# reasoning_results = await eval_model(REASONING_MODEL)
# Upgrade only if quality difference justifies the cost
'''

print("Model comparison pattern:")
print(MODEL_EVAL_TEMPLATE)

Replace the scoring comment with the judge-agent rubric pattern from Lesson 09 to produce a quality score per test case.

Cost can be read from `result.usage` on each run — sum input and output tokens across the test set, then multiply by the model's per-token rate.

---

## 🔀 Part 2: Single Agent vs Multi-Agent

Multi-agent systems are more powerful — and more complex.

The question is whether the power justifies the complexity for your specific problem.

**Stay single-agent when:**

- One agent with good tools and instructions solves the problem

- The task is sequential, not parallel

- Debugging is easier with one agent — you have one trace to read

- The added coordination overhead of multiple agents would slow you down

**Add a second agent when:**

- The task genuinely requires two different capabilities that conflict in one instruction set

- You need a critic or reviewer that catches things the producer misses (the *Debate & Critique* notebook)

- Routing is complex enough that a triage agent improves overall quality (the *Handoffs* notebook)

- Independent subtasks can run in parallel and meaningfully speed things up (the *Parallel Execution* notebook)

**The test:** can you write clear instructions for one agent that would solve this problem?

If yes, stay single.

If the instructions become contradictory or the agent can't do two things well simultaneously, that's when multi-agent earns its complexity cost.

---

## 🛠️ Part 3: Tool-Heavy vs Instruction-Heavy

Two different philosophies for building agents — and both work, for different problems.

**Tool-heavy agents** do most of their work through callable functions. Instructions are minimal — the tools handle the logic.

- Best for: data retrieval, calculations, external API calls, anything with deterministic logic

- Risk: tool proliferation — more tools means more decisions for the model, which can degrade reliability

- Rule of thumb: if the model is calling more than 5–7 tools per run, consider simplifying

**Instruction-heavy agents** do most of their work through reasoning. Tools are minimal or absent — the instructions carry the behavior.

- Best for: writing, analysis, classification, tasks where the output depends on judgment rather than data

- Risk: instructions become too long to be reliable — models follow the beginning and end of long instruction sets better than the middle

- Rule of thumb: if instructions exceed 500 words, look for places to push logic into tools or split into multiple agents

**Most real agents are a mix.** The question is where the balance sits for your specific use case.

Tools are not more "production" than instructions — a well-crafted instruction set often outperforms a tool-heavy agent on tasks that require judgment rather than data retrieval.

---

## 💾 Part 4: When Memory and Sessions Add Value

Sessions and memory solve real problems — but they also add state, and state adds complexity.

Add them only when the simpler stateless version demonstrably fails.

**Add sessions (the *Sessions & Conversation State* notebook) when:**

- Users complain that the agent forgot what they said two turns ago

- The conversation context is essential for correct answers — not just nice to have

- The interaction is genuinely conversational, not a series of independent queries

**Add persistent memory (the *Persistent Memory* notebook) when:**

- Users need preferences or facts to persist across separate sessions

- The agent is deployed to real users who interact repeatedly over time

- You have a clear plan for what to store and what to expire

Choose vector memory (the *Vector Memory* notebook) over SQLite when retrieval should be by meaning rather than exact key — useful for large freeform knowledge bases where the user's phrasing may not match what was stored.

**Skip memory when:**

- Each query is independent — memory adds overhead with no benefit

- The use case is batch processing, not conversation

- You don't yet know what information is worth remembering — build without it first

**The test:** does the agent produce wrong or incomplete answers without memory?

If no, skip it.

## 🔌 Part 5: When MCP Is Worth the Setup

MCP gives you instant access to a growing ecosystem of pre-built tool servers.

But it adds a runtime dependency, requires Node.js or `uvx` (a utility for launching Python-based MCP servers), and introduces a new failure mode (the server itself).

It's worth it in specific situations.

**Use MCP when:**

- A pre-built MCP server does exactly what you need — filesystem, web fetch, database, GitHub, etc.

- You want to swap tool providers without rewriting agent code

- The tool ecosystem matters more than the overhead of running a subprocess

**Skip MCP and use `@function_tool` when:**

- You're wrapping your own API or business logic — `@function_tool` is simpler and faster

- You need tight control over error handling, retries, or logging

- The MCP server overhead (startup time, subprocess management) isn't worth it for your use case

- You're building something simple — don't introduce MCP complexity until you need the ecosystem

**The test:** does a pre-built MCP server exist that does what you need?

If yes, MCP is likely worth it.

If you'd be writing the server yourself, use `@function_tool`.

---

## ✂️ Part 6: Avoiding Unnecessary Complexity

The most common mistake in agent development is reaching for powerful patterns before the simpler version has been tried and found insufficient.

**The sequence that works:**

1. Start with a single agent and `MODEL`
2. Evaluate it against a golden test set (the *Testing & Evaluating Agents* notebook)
3. Identify specific failures — not vibes, specific test cases that fail
4. Add the minimum complexity that fixes those failures
5. Re-evaluate — did the fix actually work?
6. Repeat

**The rule:** add complexity only when the simpler version demonstrably fails at a specific, observable task.

---

## 🗺️ Part 7: Choosing the Right Pattern

Use this as a quick reference when you're deciding which patterns to reach for:

<div style="text-align: left; display: inline-block;">

| Decision | Use when | Skip when |
|---|---|---|
| `REASONING_MODEL` | Orchestration, planning, ambiguous judgment | Well-defined tasks, latency matters |
| Multi-agent | Conflicting capabilities, parallelism, critic needed | Single agent with good instructions works |
| Tool-heavy | Deterministic logic, external data, calculations | Pure reasoning tasks |
| Sessions | Conversation context matters across turns | Independent queries |
| Persistent memory | Users interact repeatedly over time | Batch processing, single-session use |
| Guardrails | Protecting against clear, high-impact risks (e.g., off-topic requests, PII leakage) | Trying to preemptively cover every edge case before you've seen any real usage |
| Human-in-the-loop | Irreversible actions, high-stakes decisions | Routine, reversible operations |
| MCP | Pre-built server exists for your need | Wrapping your own logic |
| Parallel execution | Independent subtasks, speed matters | Sequential dependencies |

</div>

## 🎁 Part 8: Course Wrap-Up & Suggested Next Projects

You've now covered the full arc: from single-agent tool use through multi-agent systems, safety, MCP, and production structure.

The hard part was never learning what's possible — it was learning when to reach for each pattern and when to leave it on the shelf.

Pick one of these as your next build, or use a real project you care about:

- **Research paper reviewer** — agent reads papers from a folder, applies a judge-agent rubric, and outputs a ranked summary

- **Personal finance assistant** — MCP filesystem + a simple ledger tool; approval-gated for any write action

- **Meeting notes summarizer** — takes a raw transcript, extracts action items, and saves a structured summary to disk

- **Code review agent** — reads a diff or PR, runs tests, and posts a structured review; guardrail on off-topic requests

- **Customer feedback analyzer** — batch-processes feedback files, clusters themes, and outputs a prioritized report

- **Web research agent** — MCP web fetch + filesystem; approval-gated saves; streaming output so progress is visible

Start with the simplest version.

Add complexity only when you have data showing you need it.

---

## 💪 Practice Exercises

### Exercise 1: Apply the Decision Framework

*Covers: architecture decisions — choosing patterns before coding*

Use the decision framework on a real or suggested project to choose the right patterns before writing any code.

In [ ]:
# --------------------------------------------------------------
# 💪 Apply the Decision Framework
# --------------------------------------------------------------
# Objective: Use the decision framework from this notebook on a real agent idea.

# TODO 1: Pick one of the six suggested projects from Part 8 — or use
#          a real project you want to build.

# TODO 2: Work through each decision in order:
#          - Which model? (MODEL or REASONING_MODEL) — why?
#          - Single agent or multi-agent? — what is the test?
#          - Tool-heavy or instruction-heavy? — where is the balance?
#          - Sessions or memory needed? — what would trigger adding them?
#          - MCP or @function_tool? — does a pre-built server exist?
#          - What is the simplest version you could build first?

# TODO 3: Write your answers as comments below this line.
#          There is no runnable code here — this is a design exercise.
#          The goal is to reason through complexity before writing any code.

# --- Write your design decisions below this line ---

---

## 🎯 Key Takeaways

**Start simple, justify upgrades with evidence:**

- Default to `MODEL`, single agent, no sessions, no MCP

- Add complexity only when a specific, observable failure demands it

- Use your Testing & Evaluating Agents test sets to confirm upgrades actually improve quality
<br>
<br>

**Each decision has a clear threshold:**

- `REASONING_MODEL`: orchestration and planning, not general use

- Multi-agent: when one agent genuinely can't do two things well simultaneously

- Sessions: when users complain about the agent forgetting

- MCP: when a pre-built server exists for your exact need
<br>
<br>

**Complexity compounds:**

- Every pattern added is another thing to debug, monitor, and maintain

- The agents that work in production are almost always simpler than they look in demos

- Build the simplest version that solves the real problem — then stop

---

## 📍 Next Step

**Notebook 32: Deploying Agents with Gradio**  

Wrap your agent in a Gradio chat interface, stream responses live, and deploy to Hugging Face Spaces.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-31-architecture-decisions)

---